In [2]:
from bs4 import BeautifulSoup
import re

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # Remove obvious noise
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # First try common article containers
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text

# Example
text = extract_article_text("/content/24_Source_1.html")
print(text[:3000])

Fakes, Bots, and Blockings in Armenia A snapshot of online manipulation on the eve of a parliamentary vote @DFRLab Follow 5 min read · Apr 2, 2017 40 1 Source : Twitter. Accounts posting the same wording and image ahead of the Armenian election. Armenians go to the polls on April 2 for the first election under a new law which marks the transition towards a parliamentary system of rule. The days before the election saw a spike in manipulative behavior on Twitter, including fakes, bots, and the targeted blocking of key accounts. Fake e-mail First, at the end of March, tens of Twitter users tweeting in Russian started sharing a fake USAID email implying that the US is meddling in Armenia’s elections. Press enter or click to view image in full size Source : Pastebin, via Onnik J Krikorian / Twitter The email was debunked by the US Embassy in Yerevan, which pointed out grammatical and spelling mistakes in the text, and the implausibility of a genuine USAID email coming from a gmail account:

In [ ]:
import os
import re
from bs4 import BeautifulSoup

def extract_article_text(file_path):
    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        html = f.read()

    soup = BeautifulSoup(html, "html.parser")

    # ❌ Remove junk elements
    for tag in soup(["script", "style", "noscript", "header", "footer", "nav", "aside", "form"]):
        tag.decompose()

    # 🎯 Try to find main article content
    candidates = [
        soup.find("article"),
        soup.find("main"),
        soup.find(attrs={"role": "main"}),
        soup.find("div", class_=re.compile(r"article|content|post|story|body", re.I)),
    ]

    main_content = next((c for c in candidates if c is not None), soup)

    # 🧹 Extract and clean text
    text = main_content.get_text(separator=" ", strip=True)
    text = re.sub(r"\s+", " ", text)

    return text


# =========================
# 📂 Process entire folder
# =========================
input_folder = "articles_html"
output_folder = "clean_text"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

for filename in os.listdir(input_folder):
    if filename.endswith(".html"):
        file_path = os.path.join(input_folder, filename)

        try:
            clean_text = extract_article_text(file_path)

            # Save cleaned text
            output_file = os.path.join(output_folder, filename.replace(".html", ".txt"))
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(clean_text)

            print(f"✅ Processed: {filename}")

        except Exception as e:
            print(f"❌ Error with {filename}: {e}")

print("\n🎉 Done! Clean text saved in 'clean_text/' folder.")

In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os
print(os.listdir("/content/drive/MyDrive"))

['Colab Notebooks', 'Rstudio.docx.gdoc', 'roberta_model', 'NLP']


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "/content/drive/MyDrive/NLP/roberta_model"

model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

model.eval()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (

In [14]:
import torch

text = "This campaign used misinformation to influence voters"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=256
)

with torch.no_grad():
    outputs = model(**inputs)

pred_id = torch.argmax(outputs.logits, dim=1).item()
pred_label = model.config.id2label[pred_id]

print("Prediction:", pred_label)

Prediction: Adversary Action
